# CropCop Track B — 01 Final Execution

This is the **only supported Track-B scientific execution notebook** for v4.

Attach exactly two private datasets produced by `TrackB_00_Readiness_Materialization.ipynb`:
1. `cropcop-trackb-r07-infrastructure-v4`
2. `cropcop-trackb-r07-external-v4`

Pin the exact dataset versions reported by the readiness receipt.

### Qualification
- T4 x2
- Internet ON (runtime package repair only)
- **No secrets**
- `RUN_MODE = "qualification"`
- Produces zero protected external predictions and stops after independent Q3 verification.

### Claim
Use the same notebook and exact same two dataset versions only after qualification review:
- `RUN_MODE = "claim"`
- paste the reviewed `qualification_science_sha256`
- enable only `KAGGLE_API_TOKEN`
- GitHub credentials are not part of the scientific run.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

RUN_MODE = 'qualification'  # qualification | claim
AUTHORIZED_QUALIFICATION_SCIENCE_SHA256 = ''
KAGGLE_OWNER = 'AUTO'

if RUN_MODE not in {'qualification', 'claim'}:
    raise RuntimeError('RUN_MODE must be qualification or claim')
if RUN_MODE == 'qualification' and AUTHORIZED_QUALIFICATION_SCIENCE_SHA256:
    raise RuntimeError('Qualification mode must not carry a claim authorization digest')
if RUN_MODE == 'claim':
    value = AUTHORIZED_QUALIFICATION_SCIENCE_SHA256.strip().lower()
    if len(value) != 64 or any(ch not in '0123456789abcdef' for ch in value):
        raise RuntimeError('Claim mode requires the reviewed 64-character qualification science SHA-256')


In [ ]:
INPUT = Path('/kaggle/input')
bundle_files = {
    'infra': sorted(INPUT.glob('**/TRACKB_INFRASTRUCTURE_BUNDLE.json')),
    'external': sorted(INPUT.glob('**/TRACKB_EXTERNAL_BUNDLE.json')),
}
if len(bundle_files['infra']) != 1 or len(bundle_files['external']) != 1:
    raise RuntimeError(f'Attach exactly one infrastructure and one external v4 bundle: {bundle_files}')

infra = json.loads(bundle_files['infra'][0].read_text())
external = json.loads(bundle_files['external'][0].read_text())
if infra.get('materialization_id') != external.get('materialization_id'):
    raise RuntimeError('Attached v4 bundles have different materialization_id values')

roles = {}
for p in sorted(INPUT.glob('**/TRACKB_INPUT_MANIFEST.json')):
    obj = json.loads(p.read_text())
    role = obj.get('role')
    if role in roles:
        raise RuntimeError(f'Duplicate Track-B role: {role}')
    roles[role] = (p, obj)
required = {'core','historical_compare','gvlid_v5','irish_potato'}
if set(roles) != required:
    raise RuntimeError(f'Attached Track-B roles mismatch: expected={sorted(required)}, observed={sorted(roles)}')

core_path, core = roles['core']
repo_rel = str(core.get('repository_root','')).strip()
REPO = (core_path.parent / repo_rel).resolve()
if not (REPO / 'journal_extension').is_dir():
    raise RuntimeError(f'Embedded repository snapshot missing: {REPO}')

print(json.dumps({
    'materialization_id': infra['materialization_id'],
    'repository_source_sha': infra['repository_source_sha'],
    'roles': sorted(roles),
    'embedded_repo': str(REPO),
}, indent=2, sort_keys=True))


In [ ]:
WORK = Path('/kaggle/working')
BOOTSTRAP = REPO / 'journal_extension/scripts/bootstrap_trackb_runtime.py'
LOCKFILE = REPO / 'journal_extension/track_b_r07/requirements-trackb.lock.txt'
BOOTSTRAP_RECEIPT = WORK / 'TRACKB_V4_EXECUTION_RUNTIME.json'

subprocess.run([
    sys.executable, str(BOOTSTRAP),
    '--requirements', str(LOCKFILE),
    '--receipt', str(BOOTSTRAP_RECEIPT),
], cwd=REPO, check=True, env=os.environ.copy())

runtime = json.loads(BOOTSTRAP_RECEIPT.read_text())
if runtime.get('status') != 'PASS':
    raise RuntimeError('Track-B v4 runtime bootstrap failed')
if runtime.get('scientific_execution_requires_fresh_subprocess') is not True:
    raise RuntimeError('Runtime receipt does not require a fresh scientific subprocess')

print(json.dumps({
    'runtime_status': runtime['status'],
    'cuda_available': runtime['probe']['cuda_available'],
    'cuda_devices': runtime['probe']['cuda_devices'],
    'versions': runtime['probe']['versions'],
}, indent=2, sort_keys=True))


In [ ]:
if RUN_MODE == 'claim':
    from kaggle_secrets import UserSecretsClient
    token = (UserSecretsClient().get_secret('KAGGLE_API_TOKEN') or '').strip()
    if not token or any(ch.isspace() for ch in token):
        raise RuntimeError('Claim mode requires a valid KAGGLE_API_TOKEN')
    os.environ['KAGGLE_API_TOKEN'] = token
    del token
else:
    os.environ.pop('KAGGLE_API_TOKEN', None)
    os.environ.pop('CROPCOP_GITHUB_TOKEN', None)

CONTROLLER = REPO / 'journal_extension/scripts/trackb_v4_execute_attached.py'
OUT = WORK / 'trackb_r07'
SCRATCH = Path('/kaggle/tmp/cropcop_trackb_r07_v4')
if OUT.exists():
    shutil.rmtree(OUT)
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)

cmd = [
    sys.executable, str(CONTROLLER),
    '--input-root', '/kaggle/input',
    '--output-root', str(OUT),
    '--scratch-root', str(SCRATCH),
    '--device', 'cuda:0',
    '--mode', RUN_MODE,
    '--kaggle-owner', KAGGLE_OWNER,
]
if RUN_MODE == 'claim':
    cmd += ['--authorized-qualification-science-sha256', AUTHORIZED_QUALIFICATION_SCIENCE_SHA256.strip().lower()]

subprocess.run(cmd, cwd=REPO, check=True, env=os.environ.copy())


In [ ]:
receipt = json.loads((Path('/kaggle/working/trackb_r07') / 'TRACKB_V4_EXECUTION_RECEIPT.json').read_text())

if RUN_MODE == 'qualification':
    if receipt.get('status') != 'PASS_TRACKB_PREINFERENCE_QUALIFICATION':
        raise RuntimeError(f"Qualification did not PASS: {receipt.get('status')}")
    if receipt.get('protected_external_prediction_count') != 0:
        raise RuntimeError('Qualification produced protected external predictions')
    if receipt.get('v1_test_accessed') is not False:
        raise RuntimeError('Qualification reports V1-test access')
    summary = {
        'status': receipt['status'],
        'materialization_id': receipt['materialization_id'],
        'qualification_science_sha256': receipt['qualification_science_sha256'],
        'preinference_qa_sha256': receipt['preinference_qa_sha256'],
        'protected_external_prediction_count': receipt['protected_external_prediction_count'],
        'v1_test_accessed': receipt['v1_test_accessed'],
        'next_gate': 'Independent review of qualification evidence before claim authorization.',
    }
else:
    if receipt.get('status') != 'PASS_TRACKB_CLOSED_PRIVATE_ARCHIVED':
        raise RuntimeError(f"Claim did not close safely: {receipt.get('status')}")
    summary = {
        'status': receipt['status'],
        'materialization_id': receipt['materialization_id'],
        'trackb_science_sha256': receipt['trackb_science_sha256'],
        'closure_sha256': receipt['closure_sha256'],
        'final_qa_sha256': receipt['final_qa_sha256'],
        'private_evidence': receipt['private_evidence']['slug'],
        'public_evidence_zip': receipt['public_evidence_zip'],
        'next_gate': 'Commit the public-safe evidence package to the repository; do not rerun science.',
    }

print(json.dumps(summary, indent=2, sort_keys=True))
